# **Portfolio:** Mario Casanova — Data Science & Analytics Portfolio
## **Case Study:** Computational Physics & Mathematical Integrators

---
*Nota Institucional: Este notebook implementa un motor cinemático para la dispersión de partículas bajo campos de potencial coulombiano. Evitando métodos inestables como Euler, exige el uso riguroso de un Integrador Simpléctico de Verlet para procesar las trayectorias espaciales, garantizando la conservación absoluta de la energía del sistema hamiltoniano. El producto de este ejercicio numérico intensivo es la deducción estadística empírica de una sección eficaz física.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from numba import jit
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')

# Constantes físicas naturalizadas al entorno simulado
k_e = 1.0  # Constante de Coulomb escalada
Q_au = 79.0 # Carga de Oro (Núcleo objetivo)

# Discretización de tiempo
DT = 0.05
STEPS = 5000

### 1. Ingestión del Ensamble de Monte Carlo
Se carga el conjunto de condiciones iniciales (posiciones ortogonales y momentos) de 5000 partículas alfa pre-generadas.

In [ ]:
df_ensemble = pd.read_csv('../data/rutherford_initial_particles.csv')

# Extraemos arreglos numpy para acelerar Numba (Just-In-Time Compilation)
x_init = df_ensemble['x'].values
y_init = df_ensemble['y'].values
vx_init = df_ensemble['vx'].values
vy_init = df_ensemble['vy'].values
q1 = df_ensemble['charge_q1'].values[0]
m1 = df_ensemble['mass_m1'].values[0]

print(f"[*] Ensamble recuperado: {len(x_init)} partículas alfa (Carga +{q1}, Masa {m1}u). Movimiento 2D asimétrico.")

### 2. Integrador Simpléctico de Velocity-Verlet (Cálculo Vectorial Diferencial)
El algoritmo computa las fuerzas centrales en cada diferencia de tiempo $(dt)$, actualiza las posiciones, recalcula gradientes y actualiza velocidades. Se compila en Numba (`@jit`) para replicar la velocidad multihilo (in silico) que C++ o C# ofrecerían de forma nativa frente al ciclo GIL estándar de Python.

In [ ]:
@jit(nopython=True)
def compute_coulomb_force(x, y, q1_val, m1_val):
    """
    Computación vectorizada del inverso del cuadrado de la distancia.
    F = (k_e * q1 * Q_au) / r^2 en dirección radial.
    Retorna las aceleraciones ax, ay.
    """
    r_sq = x**2 + y**2
    # Prevenimos divergencias de colisión puramente centrales añadiendo factor de suavizado
    r_sq = np.where(r_sq < 1.0, 1.0, r_sq)
    r = np.sqrt(r_sq)
    
    force_mag = (k_e * q1_val * Q_au) / r_sq
    
    # Descomposición del vector radial de fuerza (repulsiva)
    ax = (force_mag / m1_val) * (x / r)
    ay = (force_mag / m1_val) * (y / r)
    
    return ax, ay

@jit(nopython=True)
def verlet_integrator(x, y, vx, vy, dt, steps, q1_val, m1_val):
    """
    Evoluciona el sistema dinámico Hamiltoniano.
    Utilizamos Velocity-Verlet para no drenar energía térmica artificialmente.
    """
    N = len(x)
    
    # Arreglos para almacenar trayectorias de un subconjunto (para gráficos de estela)
    n_traces = 50
    x_trace = np.zeros((steps, n_traces))
    y_trace = np.zeros((steps, n_traces))
    
    # Condiciones iniciales de aceleración
    ax, ay = compute_coulomb_force(x, y, q1_val, m1_val)
    
    for t in range(steps):
        # 1. Actualizar Posición: r(t+dt) = r(t) + v(t)dt + 0.5*a(t)dt^2
        x_new = x + vx * dt + 0.5 * ax * dt**2
        y_new = y + vy * dt + 0.5 * ay * dt**2
        
        # 2. Computar Nueva Aceleración a(t+dt)
        ax_new, ay_new = compute_coulomb_force(x_new, y_new, q1_val, m1_val)
        
        # 3. Actualizar Velocidad: v(t+dt) = v(t) + 0.5*(a(t) + a(t+dt))dt
        vx_new = vx + 0.5 * (ax + ax_new) * dt
        vy_new = vy + 0.5 * (ay + ay_new) * dt
        
        # Reasignación sincrónica
        x, y = x_new, y_new
        vx, vy = vx_new, vy_new
        ax, ay = ax_new, ay_new
        
        # Almacenar trazas (primeras 50 partículas)
        x_trace[t, :] = x[:n_traces]
        y_trace[t, :] = y[:n_traces]
        
    return x_trace, y_trace, vx, vy

# Ejecutar motor compilado JIT
print("[*] Integrando Ecuaciones Diferenciales Ordinarias (Verlet)...")
x_traces, y_traces, final_vx, final_vy = verlet_integrator(x_init, y_init, vx_init, vy_init, DT, STEPS, q1, m1)
print("[+] Simulación completa convergida estructuralmente.")

### 3. Visualización Cinemática: Dispersión Vectorial
Renderizamos la evolución matricial temporal del subconjunto de rastreo. La colisión retrograda (scattering angles $> 90^{\circ}$) valida la densidad del núcleo atómico.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

# Núcleo de Oro en el origen (0,0)
ax.plot(0, 0, marker='o', markersize=20, color='gold', label='Núcleo Pesado (Au)')

# Trayectorias cinemáticas usando el historial del Verlet
for i in range(x_traces.shape[1]):
    ax.plot(x_traces[:, i], y_traces[:, i], linewidth=1.2, alpha=0.6, color='cyan')

ax.set_title("Integración de Ensayo de Rutherford (Verlet Discreto)", fontsize=15, pad=15)
ax.set_xlabel("Posición Espacial X")
ax.set_ylabel("Parámetro de Impacto Y / Deflexión Radiante")
ax.set_xlim(-200, 200)
ax.set_ylim(-150, 150)
ax.grid(alpha=0.1)
ax.legend()

plt.show()

### 4. Consolidación Analítica: Distribución Geométrica
El poder masivo predictivo del ensamble requiere abstraer los miles de vectores finales en su ángulo estadístico $(\theta)$, para modelar la probabilidad de reflexión.

In [ ]:
# El ángulo de dispersión se mide con respecto a su trayectoria entrante (Eje X positivo).
# atan2(vy, vx) produce un ángulo en radianes de -pi a pi.
final_angles = np.degrees(np.arctan2(final_vy, final_vx))

fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(final_angles, bins=100, color='red', stat="density", log_scale=(False, True), ax=ax)

ax.set_title("Distribución Log-Estadística de Deflexión Angular (5k Muestras)", fontsize=14, pad=10)
ax.set_xlabel("Ángulo de Dispersión $\\theta$ (Grados)")
ax.set_ylabel("Densidad de Probabilidad (Exponencial Log)")

ax.axvline(0, color='yellow', linestyle='--', label='Dispersión Frontal nula')
# El hecho de que la cola exista en +/-180 confirma que ciertas partículas "rebotaron" retrogradamente.
plt.legend()
plt.show()